# Deploy WhisperX speech transcription on Amazon SageMaker AI

This notebook deploys the AWS Deep Learning Containers (DLC) [WhisperX image](https://aws.github.io/deep-learning-containers/whisperx/) to both Amazon SageMaker AI real-time and asynchronous endpoints. It demonstrates transcription, word-level timestamp alignment, speaker diarization, and SRT subtitle generation through the container's OpenAI-compatible multipart API.

The example uses the published `WhisperX 3.8.6` SageMaker image. It intentionally uses a short, deterministic public-domain clip so the synchronous invocation is less likely to exceed SageMaker's 60-second real-time response limit. For long recordings, use an [asynchronous endpoint](https://aws.github.io/deep-learning-containers/whisperx/deployment/sagemaker/).

> **Cost:** A GPU real-time endpoint incurs charges while it exists. Run the final cleanup cell after testing, including after a failed experiment.

## What you will do

1. Configure an execution role, AWS Region, GPU instances, AWS-provided WhisperX DLC, and the asynchronous S3 location.
2. Create and wait for isolated real-time and asynchronous endpoints.
3. Prepare a 35-second public-domain Apollo 11 clip for real-time inference, then submit the longer source recording to the asynchronous endpoint.
4. Validate and render diarized results, write an SRT file, and delete both endpoints and this run's S3 objects.

## Prerequisites

- A SageMaker Studio notebook or another Python environment with AWS credentials.
- An IAM role trusted by SageMaker AI. Set `ROLE_ARN` explicitly unless the notebook is running in SageMaker Studio and can resolve its execution role. Do not derive or guess a role ARN from STS.
- Permission to create, describe, and delete SageMaker models, endpoint configurations, and endpoints. The role also needs the standard ECR pull permissions for AWS DLCs. For the asynchronous workflow, both your notebook credentials and the SageMaker execution role need read/write access to the selected S3 bucket and prefix; creating the default bucket also requires S3 bucket and encryption/public-access-block permissions.
- A GPU instance quota for the selected instance type. This notebook defaults to `ml.g4dn.xlarge`; choose `ml.g5.2xlarge` if it is available and you need more throughput.
- Network egress from the endpoint to download the default Whisper and language-alignment weights. The notebook environment also needs egress to download the public demo audio. For a network-isolated endpoint, stage supported model assets under `/opt/ml/model` as described in the [WhisperX configuration reference](https://aws.github.io/deep-learning-containers/whisperx/configuration/).

The SageMaker image serves `POST /invocations` on port 8080. Each request is a `multipart/form-data` body containing an audio file and optional form fields. The image serializes transcription requests, so scale out with more endpoint instances instead of increasing per-container concurrency.

In [ ]:
%pip install --quiet --upgrade "boto3>=1.34.0,<2.0.0" "botocore>=1.34.0,<2.0.0" "sagemaker>=2.200.0,<3.0.0" "imageio-ffmpeg==0.6.0"

# Restart the kernel if the package manager reports that a restart is required.

## Configure the deployment

The image account ID differs in a small number of AWS Regions. The next cell resolves the documented regional private-ECR account ID and builds the supported SageMaker image URI. It also pins the GPU AMI required by the CUDA 12.8 WhisperX image.

In [ ]:
import os
import re
import uuid
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError, EndpointConnectionError, ReadTimeoutError

# Set this explicitly when not running in SageMaker Studio. Do not put credentials in this notebook.
ROLE_ARN = os.environ.get("SAGEMAKER_ROLE_ARN", "")
REGION = os.environ.get("WHISPERX_REGION", "us-west-2")
INSTANCE_TYPE = "ml.g4dn.xlarge"

# The documented SageMaker WhisperX image: Python 3.12, CUDA 12.8, Amazon Linux 2023.
IMAGE_TAG = "3.8.6-cu128-amzn2023-sagemaker"
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-gpu-3-1"
STARTUP_HEALTH_CHECK_TIMEOUT_SECONDS = 900
DEPLOY_TIMEOUT_SECONDS = 1_800
CLEANUP_TIMEOUT_SECONDS = 900

# From https://aws.github.io/deep-learning-containers/reference/region_availability/
DLC_ACCOUNT_BY_REGION = {
    "us-east-1": "763104351884", "us-east-2": "763104351884",
    "us-west-1": "763104351884", "us-west-2": "763104351884",
    "af-south-1": "626614931356", "ap-east-1": "871362719292",
    "ap-south-1": "763104351884", "ap-south-2": "772153158452",
    "ap-southeast-1": "763104351884", "ap-southeast-2": "763104351884",
    "ap-southeast-3": "907027046896", "ap-southeast-4": "457447274322",
    "ap-southeast-5": "550225433462", "ap-southeast-6": "633930458069",
    "ap-southeast-7": "590183813437", "ap-northeast-1": "763104351884",
    "ap-northeast-2": "763104351884", "ap-northeast-3": "364406365360",
    "ap-east-2": "975050140332", "ca-central-1": "763104351884",
    "ca-west-1": "204538143572", "eu-central-1": "763104351884",
    "eu-central-2": "380420809688", "eu-west-1": "763104351884",
    "eu-west-2": "763104351884", "eu-west-3": "763104351884",
    "eu-north-1": "763104351884", "eu-south-1": "692866216735",
    "eu-south-2": "503227376785", "il-central-1": "780543022126",
    "me-south-1": "217643126080", "me-central-1": "914824155844",
    "mx-central-1": "637423239942", "sa-east-1": "763104351884",
    "cn-north-1": "727897471807", "cn-northwest-1": "727897471807",
}

if REGION not in DLC_ACCOUNT_BY_REGION:
    raise ValueError(
        f"No WhisperX DLC account is configured for {REGION}. Check the DLC region-availability page."
    )

partition = boto3.session.Session().get_partition_for_region(REGION)
dns_suffix = "amazonaws.com.cn" if partition == "aws-cn" else "amazonaws.com"
DLC_ACCOUNT_ID = DLC_ACCOUNT_BY_REGION[REGION]
IMAGE_URI = f"{DLC_ACCOUNT_ID}.dkr.ecr.{REGION}.{dns_suffix}/whisperx:{IMAGE_TAG}"

# UTC timestamp plus randomness prevents collisions when the notebook is re-run.
run_id = f"{datetime.now(timezone.utc):%y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"
MODEL_NAME = f"whisperx-model-{run_id}"
ENDPOINT_CONFIG_NAME = f"whisperx-config-{run_id}"
ENDPOINT_NAME = f"whisperx-{run_id}"

sm = boto3.client("sagemaker", region_name=REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)
ecr = boto3.client("ecr", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

In [ ]:
ROLE_ARN_PATTERN = re.compile(r"^arn:[^:]+:iam::\d{12}:role/.+")

def resolve_execution_role(configured_role_arn):
    if configured_role_arn:
        return configured_role_arn
    try:
        from sagemaker import get_execution_role
        return get_execution_role()
    except Exception as error:
        raise ValueError(
            "Set ROLE_ARN (or SAGEMAKER_ROLE_ARN) to an IAM role trusted by SageMaker AI. "
            "Automatic role discovery requires the SageMaker Python SDK and a supported SageMaker environment."
        ) from error

ROLE_ARN = resolve_execution_role(ROLE_ARN)
if not ROLE_ARN_PATTERN.fullmatch(ROLE_ARN):
    raise ValueError(f"ROLE_ARN is not a valid IAM role ARN: {ROLE_ARN!r}")

# This read-only check gives an early, useful error if the tag is not published in this Region.
# Some restrictive IAM policies omit ecr:DescribeImages; endpoint creation remains the definitive pull check.
try:
    image = ecr.describe_images(
        registryId=DLC_ACCOUNT_ID,
        repositoryName="whisperx",
        imageIds=[{"imageTag": IMAGE_TAG}],
    )["imageDetails"][0]
    print(f"Verified AWS DLC image digest: {image.get('imageDigest', '<not returned>')}")
except ClientError as error:
    print(
        f"Could not verify the image with ECR ({error.response['Error']['Code']}). "
        "Continuing with the documented image URI."
    )

print(f"Region:        {REGION}")
print(f"Execution role: {ROLE_ARN}")
print(f"Image URI:     {IMAGE_URI}")
print(f"Instance type: {INSTANCE_TYPE}")
print(f"Endpoint name: {ENDPOINT_NAME}")
print("Before deploying, confirm that this instance type has endpoint quota in this Region.")

## Define the deployment lifecycle

The helper below treats endpoint cleanup as part of the deployment lifecycle. The final workflow cell creates the endpoint only after audio preparation has succeeded, then uses a `try`/`finally` block to clean up after deployment, transcription, rendering, or SRT-generation failures. Cleanup waits for the endpoint to disappear before removing the configuration and model, which prevents resource-in-use failures and reduces the chance of leaving a billable endpoint behind.

In [ ]:
import time

NOT_FOUND_CODES = {"ResourceNotFound", "ResourceNotFoundException", "ValidationException"}

def is_not_found(error):
    return error.response.get("Error", {}).get("Code") in NOT_FOUND_CODES

def wait_for_endpoint(target_status, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    last_status = None
    while time.monotonic() < deadline:
        try:
            endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        except ClientError as error:
            if target_status == "Deleted" and is_not_found(error):
                return
            raise
        status = endpoint["EndpointStatus"]
        if status != last_status:
            print(f"Endpoint status: {status}")
            last_status = status
        if status == target_status:
            return
        if status == "Failed":
            raise RuntimeError(endpoint.get("FailureReason", "Endpoint deployment failed without a reason."))
        time.sleep(15)
    raise TimeoutError(f"Endpoint did not reach {target_status} within {timeout_seconds} seconds.")

def delete_after_endpoint(label, delete_call):
    deadline = time.monotonic() + CLEANUP_TIMEOUT_SECONDS
    while True:
        try:
            delete_call()
            print(f"Deleted {label}.")
            return
        except ClientError as error:
            if is_not_found(error):
                print(f"{label} is already absent.")
                return
            if error.response.get("Error", {}).get("Code") != "ResourceInUse":
                raise
            if time.monotonic() >= deadline:
                raise TimeoutError(f"Timed out waiting to delete {label}.") from error
            time.sleep(10)

def cleanup_resources():
    errors = []
    try:
        sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
        print(f"Deleting endpoint {ENDPOINT_NAME} ...")
        wait_for_endpoint("Deleted", CLEANUP_TIMEOUT_SECONDS)
        print("Endpoint deleted.")
    except ClientError as error:
        if is_not_found(error):
            print("Endpoint is already absent.")
        else:
            errors.append(("endpoint", error))
    except Exception as error:
        errors.append(("endpoint", error))

    for label, delete_call in (
        ("endpoint configuration", lambda: sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)),
        ("model", lambda: sm.delete_model(ModelName=MODEL_NAME)),
    ):
        try:
            delete_after_endpoint(label, delete_call)
        except Exception as error:
            errors.append((label, error))

    if errors:
        details = "; ".join(f"{label}: {error}" for label, error in errors)
        raise RuntimeError(f"Cleanup was incomplete. Re-run this cell after resolving: {details}")

def deploy_endpoint():
    try:
        sm.create_model(
            ModelName=MODEL_NAME,
            PrimaryContainer={
                "Image": IMAGE_URI,
                "Environment": {"WHISPERX_DEFAULT_MODEL": "large-v2"},
            },
            ExecutionRoleArn=ROLE_ARN,
        )
        sm.create_endpoint_config(
            EndpointConfigName=ENDPOINT_CONFIG_NAME,
            ProductionVariants=[{
                "VariantName": "AllTraffic",
                "ModelName": MODEL_NAME,
                "InitialInstanceCount": 1,
                "InstanceType": INSTANCE_TYPE,
                # Required for the CUDA 12.8 WhisperX DLC.
                "InferenceAmiVersion": INFERENCE_AMI_VERSION,
                "ContainerStartupHealthCheckTimeoutInSeconds": STARTUP_HEALTH_CHECK_TIMEOUT_SECONDS,
            }],
        )
        sm.create_endpoint(
            EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME
        )
        print(f"Creating endpoint {ENDPOINT_NAME} ...")
        wait_for_endpoint("InService", DEPLOY_TIMEOUT_SECONDS)
        print("Endpoint is ready.")
    except Exception:
        print("Deployment failed. Attempting cleanup before re-raising the error ...")
        try:
            cleanup_resources()
        except Exception as cleanup_error:
            print(f"Automatic cleanup also failed: {cleanup_error}")
        raise

In [ ]:
# No endpoint is created in this cell. Continue to prepare the audio and request helpers.
# The final workflow cell deploys, invokes, writes SRT output, and cleans up in one guarded run.
print("Deployment helper is ready; no billable resources have been created.")

## Prepare a short audio sample

The default sample is a public-domain US federal government recording of the US Airways Flight 1549 air traffic control communications, hosted by Wikimedia Commons. It is a genuine multi-speaker exchange between the pilot and several controllers under real-world channel conditions, which makes it a good sample for evaluating transcription accuracy, word-level timestamps, and speaker diarization. The code uses `imageio-ffmpeg`, installed in the setup cell, to produce a deterministic 40-second 16-kHz mono WAV for the synchronous request. Set `LOCAL_AUDIO_PATH` to use your own audio. You are responsible for having the required rights and permissions for any audio you submit.

The WhisperX container accepts common audio formats, but a short WAV keeps the example payload and synchronous request predictable. The container has a 100 MiB upload limit; long audio is a better fit for asynchronous inference, so the full recording is sent to the asynchronous endpoint later in the notebook.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import urllib.request

import imageio_ffmpeg
from IPython.display import Audio, display

LOCAL_AUDIO_PATH = None  # Example: "/home/sagemaker-user/my-audio.mp3"
# Public-domain US federal (FAA/C-SPAN-style) recording of the US Airways Flight 1549
# air traffic control communications. Genuine multi-speaker audio (pilot + controllers),
# which exercises transcription, word-level timestamps, and speaker diarization.
DEMO_URL = "https://upload.wikimedia.org/wikipedia/commons/b/b5/Flight_1549_FAA_New_York_TRACON_audio_extract.ogg"
DEMO_START_SECONDS = 0
DEMO_DURATION_SECONDS = 40

workspace_tmp = Path("/tmp")
source_path = Path(LOCAL_AUDIO_PATH) if LOCAL_AUDIO_PATH else workspace_tmp / "flight1549-source.ogg"
demo_audio_path = workspace_tmp / "whisperx-flight1549-demo.wav"
warmup_audio_path = workspace_tmp / "whisperx-warmup.wav"

if not LOCAL_AUDIO_PATH:
    request = urllib.request.Request(DEMO_URL, headers={"User-Agent": "aws-whisperx-sagemaker-demo/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response, source_path.open("wb") as output:
        shutil.copyfileobj(response, output)
    print("Downloaded public-domain Flight 1549 ATC audio from Wikimedia Commons.")
elif not source_path.is_file():
    raise FileNotFoundError(f"LOCAL_AUDIO_PATH does not exist: {source_path}")

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
def transcode(source, destination, start_seconds, duration_seconds):
    command = [
        ffmpeg, "-y", "-ss", str(start_seconds), "-i", str(source),
        "-t", str(duration_seconds), "-ac", "1", "-ar", "16000", str(destination),
    ]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode:
        raise RuntimeError(f"ffmpeg could not prepare the audio:\n{completed.stderr}")

transcode(source_path, demo_audio_path, DEMO_START_SECONDS, DEMO_DURATION_SECONDS)
transcode(demo_audio_path, warmup_audio_path, 0, min(10, DEMO_DURATION_SECONDS))
if demo_audio_path.stat().st_size == 0:
    raise RuntimeError("Prepared demo audio is empty.")

print(f"Demo audio: {demo_audio_path} ({demo_audio_path.stat().st_size:,} bytes)")
display(Audio(filename=str(demo_audio_path)))

## Invoke WhisperX

The first invocation may take longer because the container initializes the Whisper model and alignment resources. The retry policy below handles only transient network and service conditions. It does not conceal input validation, access, endpoint, or container errors.

The main request asks for `verbose_json`, word-level timestamps, and diarization. Validate the structured response before rendering or using it downstream.

In [ ]:
import html
import io
import json
import mimetypes

RAW_RESPONSE_FORMATS = {"srt", "text", "vtt"}
RETRYABLE_INVOKE_CODES = {
    "InternalFailure", "InternalServerException", "ModelNotReadyException",
    "ServiceUnavailable", "ServiceUnavailableException", "ThrottlingException",
}

def build_multipart(audio_path, fields):
    boundary = uuid.uuid4().hex
    crlf = b"\r\n"
    body = io.BytesIO()
    for name, value in fields.items():
        if "\r" in name or "\n" in name or "\r" in str(value) or "\n" in str(value):
            raise ValueError("Multipart field names and values cannot contain line breaks.")
        body.write(f"--{boundary}\r\n".encode())
        body.write(f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode())
        body.write(str(value).encode())
        body.write(crlf)

    filename = Path(audio_path).name
    media_type = mimetypes.guess_type(filename)[0] or "application/octet-stream"
    body.write(f"--{boundary}\r\n".encode())
    body.write(f'Content-Disposition: form-data; name="file"; filename="{filename}"\r\n'.encode())
    body.write(f"Content-Type: {media_type}\r\n\r\n".encode())
    body.write(Path(audio_path).read_bytes())
    body.write(crlf)
    body.write(f"--{boundary}--\r\n".encode())
    return body.getvalue(), f"multipart/form-data; boundary={boundary}"

def invoke_whisperx(audio_path, fields, max_attempts=3):
    body, content_type = build_multipart(audio_path, fields)
    response_format = fields.get("response_format", "json")
    accept = "text/plain" if response_format in RAW_RESPONSE_FORMATS else "application/json"
    for attempt in range(1, max_attempts + 1):
        caught_error = None
        try:
            response = sm_runtime.invoke_endpoint(
                EndpointName=ENDPOINT_NAME, ContentType=content_type, Accept=accept, Body=body
            )
            payload = response["Body"].read().decode("utf-8")
            return payload if response_format in RAW_RESPONSE_FORMATS else json.loads(payload)
        except (ReadTimeoutError, EndpointConnectionError) as error:
            caught_error = error
            retryable = True
        except ClientError as error:
            caught_error = error
            retryable = error.response.get("Error", {}).get("Code") in RETRYABLE_INVOKE_CODES
        if not retryable or attempt == max_attempts:
            raise caught_error
        delay = 15 * attempt
        print(f"Transient invocation failure on attempt {attempt}; retrying in {delay} seconds: {caught_error}")
        time.sleep(delay)

def validate_transcription(payload):
    if not isinstance(payload, dict):
        raise TypeError("Expected a JSON object from WhisperX.")
    if not isinstance(payload.get("text"), str):
        raise ValueError("WhisperX response did not contain a text field.")
    segments = payload.get("segments", [])
    if not isinstance(segments, list):
        raise ValueError("WhisperX response segments must be a list.")
    for index, segment in enumerate(segments):
        if not isinstance(segment, dict) or not isinstance(segment.get("text", ""), str):
            raise ValueError(f"Invalid segment at index {index}.")
    return payload

In [ ]:
# Warm the same alignment and diarization path used by the primary request.
WARMUP_FIELDS = {
    "language": "en",
    "response_format": "verbose_json",
    "timestamp_granularities[]": "word",
    "diarize": "true",
    "min_speakers": "1",
    "max_speakers": "3",
}

In [ ]:
TRANSCRIPTION_FIELDS = {
    "language": "en",
    "response_format": "verbose_json",
    "timestamp_granularities[]": "word",
    "diarize": "true",
    "min_speakers": "1",
    "max_speakers": "3",
}

In [ ]:
from IPython.display import HTML, display

PALETTE = ["#2563eb", "#dc2626", "#059669", "#7c3aed", "#d97706", "#0891b2"]

def format_time(seconds):
    seconds = max(0, int(round(float(seconds))))
    return f"{seconds // 60:02d}:{seconds % 60:02d}"

def render_transcription(transcription, elapsed_seconds):
    speakers = transcription.get("speakers") or sorted({
        segment.get("speaker", "SPEAKER_UNKNOWN") for segment in transcription.get("segments", [])
    })
    colors = {speaker: PALETTE[index % len(PALETTE)] for index, speaker in enumerate(speakers)}
    rows = []
    for segment in transcription.get("segments", []):
        speaker = str(segment.get("speaker", "SPEAKER_UNKNOWN"))
        color = colors.get(speaker, "#64748b")
        timestamp = format_time(segment.get("start", 0))
        text = html.escape(str(segment.get("text", "")).strip())
        label = html.escape(speaker)
        rows.append(
            f'<div style="padding:6px 10px;margin:3px 0;border-radius:6px;background:{color}22;border-left:4px solid {color};">'
            f'<span style="font-family:monospace;color:#475569;">[{timestamp}]</span> '
            f'<span style="color:{color};font-weight:600;">{label}</span>: {text}</div>'
        )

    duration = float(transcription.get("duration", 0) or 0)
    throughput = duration / elapsed_seconds if elapsed_seconds else 0
    display(HTML(
        '<div style="padding:16px;border:1px solid #cbd5e1;border-radius:10px;font-family:system-ui;max-width:900px;">'
        f'<p><strong>{duration:.1f}s audio</strong> · {len(speakers)} detected speaker labels · {throughput:.1f}x real time for this run</p>'
        + ''.join(rows) + '</div>'
    ))
    print("Throughput varies with audio, model, endpoint hardware, and warm-cache state.")

## Configure asynchronous inference for long audio

SageMaker asynchronous inference removes the 60-second response limit of a real-time endpoint. The request body is still `multipart/form-data`, but the notebook uploads that body to Amazon S3 and invokes the endpoint by S3 URI. SageMaker writes the result or failure information to S3.

The default bucket name contains `sagemaker`, which is compatible with the S3 scope used by the managed `AmazonSageMakerFullAccess` policy. The notebook creates this bucket only when it is absent, applies all S3 Block Public Access settings, SSE-S3 default encryption, and BucketOwnerEnforced Object Ownership, and removes only this run's objects and object versions during cleanup. The bucket itself is retained for reuse.

Supply `WHISPERX_ASYNC_BUCKET` to use an existing bucket. Before using a custom bucket, restrict its policy to the notebook caller and SageMaker execution role, then enable all S3 Block Public Access settings, default SSE-S3 or SSE-KMS encryption, and BucketOwnerEnforced Object Ownership. The notebook validates these technical controls before uploading audio.

In [ ]:
from urllib.parse import urlparse

ASYNC_INSTANCE_TYPE = os.environ.get("WHISPERX_ASYNC_INSTANCE_TYPE", INSTANCE_TYPE)
ASYNC_INVOCATION_TIMEOUT_SECONDS = 3_600
ASYNC_POLL_TIMEOUT_SECONDS = ASYNC_INVOCATION_TIMEOUT_SECONDS + 300
ACCOUNT_ID = sts.get_caller_identity()["Account"]
DEFAULT_ASYNC_BUCKET = f"sagemaker-whisperx-{REGION}-{ACCOUNT_ID}"
ASYNC_BUCKET = os.environ.get("WHISPERX_ASYNC_BUCKET", DEFAULT_ASYNC_BUCKET)
USING_DEFAULT_ASYNC_BUCKET = ASYNC_BUCKET == DEFAULT_ASYNC_BUCKET
ASYNC_PREFIX = f"whisperx-async/{run_id}"
ASYNC_MODEL_NAME = f"whisperx-async-model-{run_id}"
ASYNC_ENDPOINT_CONFIG_NAME = f"whisperx-async-config-{run_id}"
ASYNC_ENDPOINT_NAME = f"whisperx-async-{run_id}"

def s3_error_code(error):
    return error.response.get("Error", {}).get("Code", "")

def require_secure_async_bucket():
    try:
        public_access = s3.get_public_access_block(Bucket=ASYNC_BUCKET)["PublicAccessBlockConfiguration"]
    except ClientError as error:
        raise RuntimeError(
            "The asynchronous bucket must enable all four S3 Block Public Access settings."
        ) from error
    required_public_access = {
        "BlockPublicAcls": True, "IgnorePublicAcls": True,
        "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
    }
    if any(public_access.get(name) is not expected for name, expected in required_public_access.items()):
        raise RuntimeError(
            "The asynchronous bucket must enable all four S3 Block Public Access settings before it can store audio or transcripts."
        )

    try:
        rules = s3.get_bucket_encryption(Bucket=ASYNC_BUCKET)["ServerSideEncryptionConfiguration"]["Rules"]
        algorithms = {rule.get("ApplyServerSideEncryptionByDefault", {}).get("SSEAlgorithm") for rule in rules}
    except ClientError as error:
        raise RuntimeError(
            "The asynchronous bucket must have default SSE-S3 or SSE-KMS encryption."
        ) from error
    if not algorithms.intersection({"AES256", "aws:kms"}):
        raise RuntimeError("The asynchronous bucket must have default SSE-S3 or SSE-KMS encryption.")

    try:
        ownership = s3.get_bucket_ownership_controls(Bucket=ASYNC_BUCKET)["OwnershipControls"]["Rules"]
    except ClientError as error:
        raise RuntimeError(
            "The asynchronous bucket must use S3 Object Ownership with BucketOwnerEnforced."
        ) from error
    if not any(rule.get("ObjectOwnership") == "BucketOwnerEnforced" for rule in ownership):
        raise RuntimeError(
            "The asynchronous bucket must use S3 Object Ownership with BucketOwnerEnforced."
        )

def harden_new_async_bucket():
    s3.put_public_access_block(
        Bucket=ASYNC_BUCKET,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )
    s3.put_bucket_encryption(
        Bucket=ASYNC_BUCKET,
        ServerSideEncryptionConfiguration={
            "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]
        },
    )
    s3.put_bucket_ownership_controls(
        Bucket=ASYNC_BUCKET,
        OwnershipControls={"Rules": [{"ObjectOwnership": "BucketOwnerEnforced"}]},
    )

def ensure_async_bucket():
    try:
        s3.head_bucket(Bucket=ASYNC_BUCKET)
        require_secure_async_bucket()
        print(f"Using existing secure async bucket: {ASYNC_BUCKET}")
        return
    except ClientError as error:
        if s3_error_code(error) not in {"404", "NoSuchBucket", "NotFound"}:
            raise RuntimeError(
                f"Cannot access async bucket {ASYNC_BUCKET}. Set WHISPERX_ASYNC_BUCKET to a bucket your caller and execution role can access."
            ) from error

    create_args = {"Bucket": ASYNC_BUCKET}
    if REGION != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**create_args)
    harden_new_async_bucket()
    require_secure_async_bucket()
    print(f"Created and secured async bucket: {ASYNC_BUCKET}")

def wait_for_named_endpoint(endpoint_name, target_status, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    last_status = None
    while time.monotonic() < deadline:
        try:
            endpoint = sm.describe_endpoint(EndpointName=endpoint_name)
        except ClientError as error:
            if target_status == "Deleted" and is_not_found(error):
                return
            raise
        status = endpoint["EndpointStatus"]
        if status != last_status:
            print(f"{endpoint_name} status: {status}")
            last_status = status
        if status == target_status:
            return
        if status == "Failed":
            raise RuntimeError(endpoint.get("FailureReason", "Endpoint deployment failed without a reason."))
        time.sleep(15)
    raise TimeoutError(f"{endpoint_name} did not reach {target_status} within {timeout_seconds} seconds.")

def deploy_async_endpoint():
    ensure_async_bucket()
    try:
        sm.create_model(
            ModelName=ASYNC_MODEL_NAME,
            PrimaryContainer={
                "Image": IMAGE_URI,
                "Environment": {"WHISPERX_DEFAULT_MODEL": "large-v2"},
            },
            ExecutionRoleArn=ROLE_ARN,
        )
        sm.create_endpoint_config(
            EndpointConfigName=ASYNC_ENDPOINT_CONFIG_NAME,
            ProductionVariants=[{
                "VariantName": "AllTraffic",
                "ModelName": ASYNC_MODEL_NAME,
                "InitialInstanceCount": 1,
                "InstanceType": ASYNC_INSTANCE_TYPE,
                "InferenceAmiVersion": INFERENCE_AMI_VERSION,
                "ContainerStartupHealthCheckTimeoutInSeconds": 1_200,
            }],
            AsyncInferenceConfig={
                "OutputConfig": {
                    "S3OutputPath": f"s3://{ASYNC_BUCKET}/{ASYNC_PREFIX}/output/",
                    "S3FailurePath": f"s3://{ASYNC_BUCKET}/{ASYNC_PREFIX}/failure/",
                },
                "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 1},
            },
        )
        sm.create_endpoint(
            EndpointName=ASYNC_ENDPOINT_NAME, EndpointConfigName=ASYNC_ENDPOINT_CONFIG_NAME
        )
        print(f"Creating asynchronous endpoint {ASYNC_ENDPOINT_NAME} ...")
        wait_for_named_endpoint(ASYNC_ENDPOINT_NAME, "InService", DEPLOY_TIMEOUT_SECONDS)
        print("Asynchronous endpoint is ready.")
    except Exception:
        print("Asynchronous deployment failed. Attempting cleanup before re-raising the error ...")
        try:
            cleanup_async_resources()
        except Exception as cleanup_error:
            print(f"Automatic async cleanup also failed: {cleanup_error}")
        raise

def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path.lstrip("/"):
        raise ValueError(f"Invalid S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

def get_s3_object_if_available(uri):
    bucket, key = split_s3_uri(uri)
    try:
        return s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    except ClientError as error:
        if s3_error_code(error) in {"404", "NoSuchKey", "NotFound"}:
            return None
        raise

def invoke_async_whisperx(audio_path, fields):
    body, content_type = build_multipart(audio_path, fields)
    max_upload_bytes = 100 * 1024 * 1024
    if len(body) > max_upload_bytes:
        raise ValueError(
            f"The multipart request is {len(body):,} bytes; the WhisperX DLC limit is {max_upload_bytes:,} bytes."
        )
    input_key = f"{ASYNC_PREFIX}/input/{uuid.uuid4().hex}.multipart"
    s3.put_object(
        Bucket=ASYNC_BUCKET, Key=input_key, Body=body, ContentType=content_type,
        ServerSideEncryption="AES256",
    )
    response = sm_runtime.invoke_endpoint_async(
        EndpointName=ASYNC_ENDPOINT_NAME,
        InputLocation=f"s3://{ASYNC_BUCKET}/{input_key}",
        ContentType=content_type,
        InvocationTimeoutSeconds=ASYNC_INVOCATION_TIMEOUT_SECONDS,
    )
    output_uri = response["OutputLocation"]
    failure_uri = response.get("FailureLocation")
    deadline = time.monotonic() + ASYNC_POLL_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        payload = get_s3_object_if_available(output_uri)
        if payload is not None:
            return json.loads(payload.decode("utf-8"))
        if failure_uri:
            failure = get_s3_object_if_available(failure_uri)
            if failure is not None:
                raise RuntimeError(f"Asynchronous inference failed: {failure.decode('utf-8', 'replace')}")
        time.sleep(10)
    raise TimeoutError(f"No asynchronous inference result after {ASYNC_POLL_TIMEOUT_SECONDS} seconds.")

def delete_async_prefix():
    try:
        versioning = s3.get_bucket_versioning(Bucket=ASYNC_BUCKET).get("Status")
        keys = []
        if versioning in {"Enabled", "Suspended"}:
            paginator = s3.get_paginator("list_object_versions")
            for page in paginator.paginate(Bucket=ASYNC_BUCKET, Prefix=f"{ASYNC_PREFIX}/"):
                for item in page.get("Versions", []) + page.get("DeleteMarkers", []):
                    keys.append({"Key": item["Key"], "VersionId": item["VersionId"]})
        else:
            paginator = s3.get_paginator("list_objects_v2")
            for page in paginator.paginate(Bucket=ASYNC_BUCKET, Prefix=f"{ASYNC_PREFIX}/"):
                keys.extend({"Key": item["Key"]} for item in page.get("Contents", []))
    except ClientError as error:
        if s3_error_code(error) in {"404", "NoSuchBucket", "NotFound"}:
            print("Asynchronous S3 bucket is already absent.")
            return
        raise

    deleted = 0
    for start in range(0, len(keys), 1_000):
        response = s3.delete_objects(
            Bucket=ASYNC_BUCKET, Delete={"Objects": keys[start:start + 1_000], "Quiet": False}
        )
        errors = response.get("Errors", [])
        if errors:
            details = "; ".join(f"{item.get('Key')}: {item.get('Code')}" for item in errors)
            raise RuntimeError(f"S3 did not delete all asynchronous run objects: {details}")
        deleted += len(response.get("Deleted", []))
    if keys:
        print(f"Deleted {deleted} S3 objects or versions for this async run.")

def cleanup_async_resources():
    errors = []
    try:
        sm.delete_endpoint(EndpointName=ASYNC_ENDPOINT_NAME)
        print(f"Deleting asynchronous endpoint {ASYNC_ENDPOINT_NAME} ...")
        wait_for_named_endpoint(ASYNC_ENDPOINT_NAME, "Deleted", CLEANUP_TIMEOUT_SECONDS)
        print("Asynchronous endpoint deleted.")
    except ClientError as error:
        if is_not_found(error):
            print("Asynchronous endpoint is already absent.")
        else:
            errors.append(("asynchronous endpoint", error))
    except Exception as error:
        errors.append(("asynchronous endpoint", error))

    for label, delete_call in (
        ("asynchronous endpoint configuration", lambda: sm.delete_endpoint_config(EndpointConfigName=ASYNC_ENDPOINT_CONFIG_NAME)),
        ("asynchronous model", lambda: sm.delete_model(ModelName=ASYNC_MODEL_NAME)),
    ):
        try:
            delete_after_endpoint(label, delete_call)
        except Exception as error:
            errors.append((label, error))
    try:
        delete_async_prefix()
    except Exception as error:
        errors.append(("asynchronous S3 objects", error))

    if errors:
        details = "; ".join(f"{label}: {error}" for label, error in errors)
        raise RuntimeError(f"Asynchronous cleanup was incomplete. Re-run this cell after resolving: {details}")

print(f"Async endpoint: {ASYNC_ENDPOINT_NAME}")
print(f"Async bucket:   {ASYNC_BUCKET}")
print(f"Async instance: {ASYNC_INSTANCE_TYPE}")

## Deploy both endpoints, transcribe, generate SRT, and clean up

This cell creates the real-time endpoint and an asynchronous endpoint only after all setup and audio-preparation cells have succeeded. It verifies the real-time path with the short WAV, then submits the longer source audio to the asynchronous endpoint. The asynchronous request is not constrained by the real-time 60-second response cap; this notebook explicitly allows up to one hour of SageMaker processing time and polls for five additional minutes. The cell validates and renders both responses, writes real-time SRT subtitles to `/tmp/whisperx_demo.srt`, and always starts cleanup for both endpoints and this run's S3 objects. If it is interrupted, the `finally` block still attempts cleanup. The following cleanup cell remains available as a recovery action if the kernel terminates before this cell can complete.

In [ ]:
workflow_error = None
try:
    deploy_endpoint()
    deploy_async_endpoint()

    warmup_started = time.perf_counter()
    warmup_response = invoke_whisperx(warmup_audio_path, WARMUP_FIELDS)
    validate_transcription(warmup_response)
    print(f"Warm-up completed in {time.perf_counter() - warmup_started:.1f} seconds.")

    started = time.perf_counter()
    transcription = invoke_whisperx(demo_audio_path, TRANSCRIPTION_FIELDS)
    elapsed_seconds = time.perf_counter() - started
    validate_transcription(transcription)
    print(f"Transcription completed in {elapsed_seconds:.1f} seconds.")
    print(f"Received {len(transcription.get('segments', []))} segments and {len(transcription.get('words', []))} words.")
    render_transcription(transcription, elapsed_seconds)

    srt = invoke_whisperx(
        demo_audio_path,
        {
            "response_format": "srt",
            "highlight_words": "true",
            "max_line_width": "42",
        },
    )
    if not isinstance(srt, str) or not srt.strip():
        raise ValueError("WhisperX returned an empty SRT response.")
    srt_path = Path("/tmp/whisperx_demo.srt")
    srt_path.write_text(srt, encoding="utf-8")
    print(f"Wrote {srt_path} ({srt_path.stat().st_size:,} bytes).")
    print("\n".join(srt.splitlines()[:12]))

    # Submit the original (longer) audio through the asynchronous endpoint.
    async_started = time.perf_counter()
    async_transcription = invoke_async_whisperx(source_path, TRANSCRIPTION_FIELDS)
    async_elapsed_seconds = time.perf_counter() - async_started
    validate_transcription(async_transcription)
    print(f"Asynchronous transcription completed in {async_elapsed_seconds:.1f} seconds.")
    print(f"Async response contains {len(async_transcription.get('segments', []))} segments and {len(async_transcription.get('words', []))} words.")
    render_transcription(async_transcription, async_elapsed_seconds)
except BaseException as error:
    workflow_error = error
    raise
finally:
    cleanup_errors = []
    for cleanup in (cleanup_async_resources, cleanup_resources):
        try:
            cleanup()
        except Exception as cleanup_error:
            cleanup_errors.append(cleanup_error)
    if cleanup_errors:
        message = "Cleanup was incomplete: " + "; ".join(map(str, cleanup_errors))
        if workflow_error is None:
            raise RuntimeError(message)
        print(f"{message} after workflow failure.")
    else:
        print("Cleanup completed.")

## Clean up

Run this cell after an interrupted workflow. It is safe to re-run, waits for both endpoint deletions before deleting dependent resources, removes only the S3 objects created for this asynchronous run, and reports incomplete cleanup rather than silently suppressing errors. It retains the asynchronous S3 bucket for reuse.

In [ ]:
cleanup_errors = []
for cleanup in (cleanup_async_resources, cleanup_resources):
    try:
        cleanup()
    except Exception as error:
        cleanup_errors.append(error)
if cleanup_errors:
    raise RuntimeError("Cleanup was incomplete: " + "; ".join(map(str, cleanup_errors)))
print("Cleanup completed.")

## References

- [WhisperX DLC overview](https://aws.github.io/deep-learning-containers/whisperx/)
- [WhisperX on Amazon SageMaker AI](https://aws.github.io/deep-learning-containers/whisperx/deployment/sagemaker/)
- [WhisperX configuration and request fields](https://aws.github.io/deep-learning-containers/whisperx/configuration/)
- [Available AWS Deep Learning Container images](https://aws.github.io/deep-learning-containers/reference/available_images/)
- [DLC Region availability](https://aws.github.io/deep-learning-containers/reference/region_availability/)

This notebook was prepared using the published WhisperX 3.8.6 DLC documentation. Re-check the available-images page before publishing a future revision or changing the image tag.